# Part 2A: Production RAG Concepts

Interactive companion to `part2a-production-rag-concepts.md`.

**This notebook:** Conceptual frameworks and decision logic (no heavy dependencies).

**Demo notebooks:** Hands-on implementations with actual retrieval, chunking, and generation.

| Demo Notebook | Focus | Setup |
|---------------|-------|-------|
| `chunking_strategies_demo.ipynb` | Chunking comparison | Ollama |
| `late_chunking_demo.ipynb` | Late chunking | Ollama (nomic-embed) |
| `hybrid_search_demo.ipynb` | BM25 + Dense + RRF | Ollama |
| `reranking_demo.ipynb` | Cross-encoder reranking | Local models |
| `embedding_models_demo.ipynb` | Embedding comparison | Ollama |
| `qdrant_demo.ipynb` | Qdrant patterns | Qdrant (in-memory) |
| `pgvector_demo.ipynb` | pgvector patterns | Docker Compose |
| `haystack_rag_demo.ipynb` | Haystack 2.x RAG | Ollama |
| `langchain_rag_demo.ipynb` | LangChain LCEL RAG | Ollama |
| `litellm_demo.ipynb` | Provider switching | Ollama |

---

## 1. RAG Failure Mode Analysis

Before choosing an architecture, understand what breaks in naive RAG.

In [30]:
# RAG Failure Modes and Solutions Mapping
#
# Use this to diagnose issues in your RAG system and select mitigations.

FAILURE_MODES = {
    "vocabulary_gap": {
        "symptom": "User says 'WFH', doc says 'remote work' → retrieval miss",
        "root_cause": "Embeddings don't bridge slang/acronyms to formal terms",
        "quick_fix": "hybrid_search",
        "thorough_fix": "hyde_expansion",
        "latency_cost": "+200-500ms",
        "demo_notebook": "hybrid_search_demo.ipynb"
    },
    "position_sensitivity": {
        "symptom": "Critical info in middle of context gets ignored by LLM",
        "root_cause": "LLMs attend more to beginning/end of long contexts",
        "quick_fix": "reduce_k_to_5",
        "thorough_fix": "reranking",
        "latency_cost": "+100-300ms",
        "demo_notebook": "reranking_demo.ipynb"
    },
    "no_relevance_check": {
        "symptom": "Query about 'AI policy' retrieves 'AI-powered HVAC' doc",
        "root_cause": "Vector similarity matched keyword, missed semantic intent",
        "quick_fix": "similarity_threshold",
        "thorough_fix": "cross_encoder_verification",
        "latency_cost": "+200-400ms",
        "demo_notebook": "reranking_demo.ipynb"
    },
    "single_pass_limitation": {
        "symptom": "'Compare X to Y' query only retrieves X, not Y",
        "root_cause": "Complex queries need multiple retrieval passes",
        "quick_fix": "multi_query_expansion",
        "thorough_fix": "agentic_rag",
        "latency_cost": "+1-5 seconds",
        "demo_notebook": "hybrid_search_demo.ipynb"
    }
}

def diagnose_failure(symptom_keywords: list[str]) -> list[dict]:
    """Find matching failure modes based on symptom keywords."""
    matches = []
    for mode, info in FAILURE_MODES.items():
        if any(kw.lower() in info["symptom"].lower() for kw in symptom_keywords):
            matches.append({"mode": mode, **info})
    return matches

# Example: User reports "retrieval misses relevant documents"
results = diagnose_failure(["retrieval", "miss"])
for r in results:
    print(f"Possible issue: {r['mode']}")
    print(f"  Quick fix: {r['quick_fix']}")
    print(f"  See: {r['demo_notebook']}\n")

Possible issue: vocabulary_gap
  Quick fix: hybrid_search
  See: hybrid_search_demo.ipynb



---

## 2. RAG Architecture Selection Framework

Match architecture complexity to actual requirements.

In [31]:
from enum import Enum
from dataclasses import dataclass

class QueryComplexity(Enum):
    SIMPLE = "simple"          # Single fact lookup
    MODERATE = "moderate"      # Requires synthesis across chunks
    COMPLEX = "complex"        # Multi-step reasoning
    AGENTIC = "agentic"        # Requires tool use beyond retrieval

class RAGArchitecture(Enum):
    NAIVE = "naive"            # embed → search → generate
    ADVANCED = "advanced"      # + query transform, reranking
    MODULAR = "modular"        # + dynamic routing
    AGENTIC = "agentic"        # + autonomous retrieval decisions

@dataclass
class RAGRequirements:
    """Capture requirements to select appropriate RAG architecture."""
    query_complexity: QueryComplexity
    daily_volume: int
    latency_budget_ms: int
    accuracy_requirement: float  # 0.0-1.0
    multi_domain: bool
    
    def recommended_architecture(self) -> RAGArchitecture:
        """
        Select SIMPLEST architecture that meets constraints.
        Complexity has costs: maintenance, latency, debugging.
        """
        # Agentic: Only if complexity demands AND latency allows
        if self.query_complexity == QueryComplexity.AGENTIC:
            if self.latency_budget_ms >= 3000:
                return RAGArchitecture.AGENTIC
            return RAGArchitecture.MODULAR  # Agentic too slow
        
        # Complex queries need sophisticated retrieval
        if self.query_complexity == QueryComplexity.COMPLEX:
            if self.multi_domain:
                return RAGArchitecture.MODULAR
            return RAGArchitecture.ADVANCED
        
        # High volume simple queries: optimize for throughput
        if self.query_complexity == QueryComplexity.SIMPLE:
            if self.daily_volume > 10000:
                return RAGArchitecture.NAIVE  # + caching
            return RAGArchitecture.ADVANCED
        
        return RAGArchitecture.ADVANCED
    
    def explain(self) -> str:
        arch = self.recommended_architecture()
        explanations = {
            RAGArchitecture.NAIVE: 
                "High volume + simple = optimize for speed. Add semantic caching.",
            RAGArchitecture.ADVANCED:
                "Moderate complexity benefits from query transform + reranking.",
            RAGArchitecture.MODULAR:
                "Multi-domain or complex queries need intelligent routing.",
            RAGArchitecture.AGENTIC:
                "Multi-step reasoning with acceptable latency. Agent decides retrieval."
        }
        return f"{arch.value.upper()}: {explanations[arch]}"

In [32]:
# Test with different scenarios

scenarios = [
    ("FAQ Bot (High Volume)", RAGRequirements(
        query_complexity=QueryComplexity.SIMPLE,
        daily_volume=50000,
        latency_budget_ms=500,
        accuracy_requirement=0.85,
        multi_domain=False
    )),
    ("Policy Search (Enterprise)", RAGRequirements(
        query_complexity=QueryComplexity.MODERATE,
        daily_volume=1000,
        latency_budget_ms=2000,
        accuracy_requirement=0.95,
        multi_domain=False
    )),
    ("Research Assistant", RAGRequirements(
        query_complexity=QueryComplexity.COMPLEX,
        daily_volume=500,
        latency_budget_ms=5000,
        accuracy_requirement=0.98,
        multi_domain=True
    )),
    ("Legal Document Analysis", RAGRequirements(
        query_complexity=QueryComplexity.AGENTIC,
        daily_volume=100,
        latency_budget_ms=10000,
        accuracy_requirement=0.99,
        multi_domain=True
    )),
]

print("RAG Architecture Selection")
print("=" * 60)
for name, req in scenarios:
    print(f"\n{name}:")
    print(f"  {req.explain()}")

RAG Architecture Selection

FAQ Bot (High Volume):
  NAIVE: High volume + simple = optimize for speed. Add semantic caching.

Policy Search (Enterprise):
  ADVANCED: Moderate complexity benefits from query transform + reranking.

Research Assistant:
  MODULAR: Multi-domain or complex queries need intelligent routing.

Legal Document Analysis:
  AGENTIC: Multi-step reasoning with acceptable latency. Agent decides retrieval.


---

## 3. Chunking Strategy Selection

Match chunking strategy to document characteristics.

In [33]:
from enum import Enum
from dataclasses import dataclass

class ChunkingStrategy(Enum):
    FIXED = "fixed_size"                # Simple, fast
    RECURSIVE = "recursive"             # Better boundaries
    HIERARCHICAL = "hierarchical"       # Preserves structure
    LATE = "late_chunking"              # Embed first, chunk later
    CONTEXTUAL = "contextual_retrieval" # LLM-enriched chunks

@dataclass
class DocumentProfile:
    """Profile a document to select optimal chunking."""
    has_structure: bool          # Headers, sections
    total_tokens: int
    high_value: bool             # Legal, medical, compliance
    chunk_count_estimate: int
    indexing_budget_per_doc: float  # $ tolerance

def select_chunking(profile: DocumentProfile) -> tuple[ChunkingStrategy, str]:
    """
    Select optimal chunking strategy.
    Returns (strategy, reasoning).
    """
    # Well-structured: preserve that structure
    if profile.has_structure:
        return (ChunkingStrategy.HIERARCHICAL, 
                "Document has clear sections. Preserve structure for context.")
    
    # Small documents: late chunking captures full context
    if profile.total_tokens < 8000:
        return (ChunkingStrategy.LATE,
                "Fits in context window. Late chunking preserves cross-references.")
    
    # High-value + budget: invest in maximum quality
    if profile.high_value and profile.chunk_count_estimate < 100000:
        if profile.indexing_budget_per_doc >= 0.001:
            return (ChunkingStrategy.CONTEXTUAL,
                    "High-value doc. LLM enrichment worth the indexing cost.")
    
    # Default: recursive with proven settings
    return (ChunkingStrategy.RECURSIVE,
            "Default choice. 400 tokens, 0 overlap. Best balance.")

In [34]:
# Test chunking selection

doc_profiles = [
    ("HR Policy Handbook", DocumentProfile(
        has_structure=True, total_tokens=5000,
        high_value=True, chunk_count_estimate=20,
        indexing_budget_per_doc=0.01
    )),
    ("Legal Contract", DocumentProfile(
        has_structure=False, total_tokens=15000,
        high_value=True, chunk_count_estimate=50,
        indexing_budget_per_doc=0.05
    )),
    ("Product Reviews (1M docs)", DocumentProfile(
        has_structure=False, total_tokens=500,
        high_value=False, chunk_count_estimate=1000000,
        indexing_budget_per_doc=0.0001
    )),
    ("API Documentation", DocumentProfile(
        has_structure=True, total_tokens=3000,
        high_value=False, chunk_count_estimate=15,
        indexing_budget_per_doc=0.001
    )),
]

print("Chunking Strategy Selection")
print("=" * 60)
for name, profile in doc_profiles:
    strategy, reasoning = select_chunking(profile)
    print(f"\n{name}:")
    print(f"  Strategy: {strategy.value}")
    print(f"  Why: {reasoning}")

Chunking Strategy Selection

HR Policy Handbook:
  Strategy: hierarchical
  Why: Document has clear sections. Preserve structure for context.

Legal Contract:
  Strategy: contextual_retrieval
  Why: High-value doc. LLM enrichment worth the indexing cost.

Product Reviews (1M docs):
  Strategy: late_chunking
  Why: Fits in context window. Late chunking preserves cross-references.

API Documentation:
  Strategy: hierarchical
  Why: Document has clear sections. Preserve structure for context.


---

## 4. Retrieval Strategy Decision

When to use dense, sparse, or hybrid search.

In [35]:
from dataclasses import dataclass
from typing import Literal

@dataclass
class QueryProfile:
    """Analyze query to select retrieval strategy."""
    has_identifiers: bool      # Error codes, SKUs, proper nouns
    has_exact_phrases: bool    # Quoted phrases
    is_conceptual: bool        # "how to", "explain", "what is"
    vocabulary_gap_likely: bool # User slang vs formal docs

def select_retrieval(
    profile: QueryProfile
) -> tuple[Literal["dense", "sparse", "hybrid"], list[str], str]:
    """
    Select retrieval strategy based on query characteristics.
    Returns (strategy, enhancements, reasoning).
    """
    enhancements = ["reranking"]  # Always recommend reranking
    
    # Identifiers or exact phrases: sparse helps a lot
    if profile.has_identifiers or profile.has_exact_phrases:
        return ("hybrid", enhancements,
                "Identifiers/exact phrases need BM25 precision + dense meaning.")
    
    # Vocabulary gap: need query transformation
    if profile.vocabulary_gap_likely:
        enhancements.append("hyde")
        enhancements.append("multi_query")
        return ("hybrid", enhancements,
                "Vocabulary gap likely. Use HyDE and multi-query expansion.")
    
    # Pure conceptual: dense is fine
    if profile.is_conceptual and not profile.has_identifiers:
        return ("dense", enhancements,
                "Conceptual query. Dense search captures semantic meaning.")
    
    # Default: hybrid is safest
    return ("hybrid", enhancements,
            "Default to hybrid. Catches both exact and semantic matches.")

In [36]:
# Test retrieval selection

queries = [
    ("Error code TS-7492", QueryProfile(
        has_identifiers=True, has_exact_phrases=False,
        is_conceptual=False, vocabulary_gap_likely=False
    )),
    ("How do I improve query performance?", QueryProfile(
        has_identifiers=False, has_exact_phrases=False,
        is_conceptual=True, vocabulary_gap_likely=False
    )),
    ("WFH rules for contractors", QueryProfile(
        has_identifiers=False, has_exact_phrases=False,
        is_conceptual=False, vocabulary_gap_likely=True
    )),
    ('Find "annual leave policy" section', QueryProfile(
        has_identifiers=False, has_exact_phrases=True,
        is_conceptual=False, vocabulary_gap_likely=False
    )),
]

print("Retrieval Strategy Selection")
print("=" * 60)
for query_text, profile in queries:
    strategy, enhancements, reasoning = select_retrieval(profile)
    print(f"\nQuery: \"{query_text}\"")
    print(f"  Strategy: {strategy}")
    print(f"  Enhancements: {', '.join(enhancements)}")
    print(f"  Why: {reasoning}")

Retrieval Strategy Selection

Query: "Error code TS-7492"
  Strategy: hybrid
  Enhancements: reranking
  Why: Identifiers/exact phrases need BM25 precision + dense meaning.

Query: "How do I improve query performance?"
  Strategy: dense
  Enhancements: reranking
  Why: Conceptual query. Dense search captures semantic meaning.

Query: "WFH rules for contractors"
  Strategy: hybrid
  Enhancements: reranking, hyde, multi_query
  Why: Vocabulary gap likely. Use HyDE and multi-query expansion.

Query: "Find "annual leave policy" section"
  Strategy: hybrid
  Enhancements: reranking
  Why: Identifiers/exact phrases need BM25 precision + dense meaning.


---

## 5. Reciprocal Rank Fusion (RRF)

The standard algorithm for merging ranked lists from different retrieval systems.

In [37]:
def reciprocal_rank_fusion(
    result_lists: list[list[dict]], 
    k: int = 60
) -> list[dict]:
    """
    Merge multiple ranked result lists using RRF.
    
    Formula: RRF(d) = Σ 1/(k + rank(d))
    
    Args:
        result_lists: List of ranked doc lists from different systems.
                      Each doc must have an 'id' field.
        k: Ranking constant (default 60 per original paper).
           Higher k = less penalty for lower ranks.
    
    Returns:
        Merged list sorted by RRF score (highest first).
    """
    score_dict = {}
    
    for list_idx, results in enumerate(result_lists):
        for rank, doc in enumerate(results):
            doc_id = doc.get('id', str(doc))
            
            if doc_id not in score_dict:
                score_dict[doc_id] = {
                    'doc': doc,
                    'score': 0.0,
                    'sources': []
                }
            
            contribution = 1.0 / (k + rank)
            score_dict[doc_id]['score'] += contribution
            score_dict[doc_id]['sources'].append({
                'list': list_idx,
                'rank': rank,
                'contribution': contribution
            })
    
    merged = sorted(
        score_dict.values(),
        key=lambda x: x['score'],
        reverse=True
    )
    
    return merged

In [38]:
# Demo: Merging dense and sparse results

dense_results = [
    {'id': 'doc_A', 'title': 'Remote Work Policy'},
    {'id': 'doc_C', 'title': 'Flexible Hours Guide'},
    {'id': 'doc_B', 'title': 'WFH Guidelines'},
    {'id': 'doc_D', 'title': 'Office Attendance'},
]

sparse_results = [
    {'id': 'doc_B', 'title': 'WFH Guidelines'},
    {'id': 'doc_A', 'title': 'Remote Work Policy'},
    {'id': 'doc_D', 'title': 'Office Attendance'},
    {'id': 'doc_E', 'title': 'Contractor Policies'},
]

merged = reciprocal_rank_fusion([dense_results, sparse_results])

print("RRF Fusion Results")
print("=" * 60)
print("\nDense:  doc_A(0), doc_C(1), doc_B(2), doc_D(3)")
print("Sparse: doc_B(0), doc_A(1), doc_D(2), doc_E(3)")
print("\nMerged (k=60):")
for i, item in enumerate(merged):
    doc = item['doc']
    score = item['score']
    sources = item['sources']
    source_str = ", ".join(f"list{s['list']}@rank{s['rank']}" for s in sources)
    print(f"  {i+1}. {doc['title']} (score: {score:.4f}) [{source_str}]")

RRF Fusion Results

Dense:  doc_A(0), doc_C(1), doc_B(2), doc_D(3)
Sparse: doc_B(0), doc_A(1), doc_D(2), doc_E(3)

Merged (k=60):
  1. Remote Work Policy (score: 0.0331) [list0@rank0, list1@rank1]
  2. WFH Guidelines (score: 0.0328) [list0@rank2, list1@rank0]
  3. Office Attendance (score: 0.0320) [list0@rank3, list1@rank2]
  4. Flexible Hours Guide (score: 0.0164) [list0@rank1]
  5. Contractor Policies (score: 0.0159) [list1@rank3]


---

## 6. Vector Database Selection

Choose based on infrastructure and requirements.

In [39]:
from dataclasses import dataclass
from typing import Literal

@dataclass
class VectorDBRequirements:
    """Capture requirements for vector database selection."""
    existing_postgres: bool
    vector_count: int
    complex_filtering: bool
    performance_critical: bool
    self_host_required: bool

def select_vector_db(
    req: VectorDBRequirements
) -> tuple[Literal["qdrant", "pgvector"], str]:
    """
    Select vector database based on requirements.
    Returns (database, reasoning).
    """
    # Large scale: need purpose-built
    if req.vector_count > 100_000_000:
        return ("qdrant", 
                ">100M vectors. Purpose-built DB handles scale better.")
    
    # Complex filtering: Qdrant's ACORN algorithm
    if req.complex_filtering:
        return ("qdrant",
                "Complex filtering needs ACORN (97% vs 53% accuracy).")
    
    # Performance critical
    if req.performance_critical:
        return ("qdrant",
                "Performance critical. Qdrant: 626 QPS at 99.5% recall.")
    
    # Existing Postgres: simplify ops
    if req.existing_postgres and req.vector_count < 100_000_000:
        return ("pgvector",
                "Existing Postgres + <100M vectors. Single DB simplifies ops.")
    
    # Default: Qdrant for greenfield
    return ("qdrant",
            "Greenfield project. Purpose-built gives best experience.")

In [40]:
# Test vector DB selection

scenarios = [
    ("Startup (greenfield)", VectorDBRequirements(
        existing_postgres=False, vector_count=1_000_000,
        complex_filtering=False, performance_critical=True,
        self_host_required=False
    )),
    ("Enterprise (existing PG)", VectorDBRequirements(
        existing_postgres=True, vector_count=10_000_000,
        complex_filtering=False, performance_critical=False,
        self_host_required=True
    )),
    ("Multi-tenant SaaS", VectorDBRequirements(
        existing_postgres=True, vector_count=50_000_000,
        complex_filtering=True, performance_critical=True,
        self_host_required=False
    )),
]

print("Vector Database Selection")
print("=" * 60)
for name, req in scenarios:
    db, reasoning = select_vector_db(req)
    print(f"\n{name}:")
    print(f"  Database: {db}")
    print(f"  Why: {reasoning}")

Vector Database Selection

Startup (greenfield):
  Database: qdrant
  Why: Performance critical. Qdrant: 626 QPS at 99.5% recall.

Enterprise (existing PG):
  Database: pgvector
  Why: Existing Postgres + <100M vectors. Single DB simplifies ops.

Multi-tenant SaaS:
  Database: qdrant
  Why: Complex filtering needs ACORN (97% vs 53% accuracy).


---

## 7. Multi-Tenancy Pattern Selection

In [41]:
from dataclasses import dataclass
from typing import Literal

@dataclass
class TenancyRequirements:
    """Requirements for multi-tenancy design."""
    tenant_count: int
    vectors_per_tenant: int
    compliance_isolation: bool  # HIPAA, SOX, etc.
    noisy_neighbor_concern: bool

def select_tenancy_pattern(
    req: TenancyRequirements
) -> tuple[Literal["payload", "collection", "database"], str]:
    """
    Select multi-tenancy isolation pattern.
    Returns (pattern, reasoning).
    """
    # Compliance mandates full isolation
    if req.compliance_isolation:
        return ("database",
                "Regulatory compliance requires full isolation.")
    
    # Large tenants with noisy neighbor concerns
    if req.vectors_per_tenant > 1_000_000 or req.noisy_neighbor_concern:
        if req.tenant_count < 500:
            return ("collection",
                    "Large tenants need index isolation. Collection per tenant.")
    
    # Default: payload filtering
    return ("payload",
            "Start simple. Payload filtering scales to thousands of tenants.")

# Example
patterns = [
    ("SaaS Platform (1000+ tenants)", TenancyRequirements(
        tenant_count=1000, vectors_per_tenant=10000,
        compliance_isolation=False, noisy_neighbor_concern=False
    )),
    ("Healthcare Platform", TenancyRequirements(
        tenant_count=50, vectors_per_tenant=100000,
        compliance_isolation=True, noisy_neighbor_concern=False
    )),
    ("Enterprise (large tenants)", TenancyRequirements(
        tenant_count=100, vectors_per_tenant=5_000_000,
        compliance_isolation=False, noisy_neighbor_concern=True
    )),
]

print("Multi-Tenancy Pattern Selection")
print("=" * 60)
for name, req in patterns:
    pattern, reasoning = select_tenancy_pattern(req)
    print(f"\n{name}:")
    print(f"  Pattern: {pattern}_per_tenant")
    print(f"  Why: {reasoning}")

Multi-Tenancy Pattern Selection

SaaS Platform (1000+ tenants):
  Pattern: payload_per_tenant
  Why: Start simple. Payload filtering scales to thousands of tenants.

Healthcare Platform:
  Pattern: database_per_tenant
  Why: Regulatory compliance requires full isolation.

Enterprise (large tenants):
  Pattern: collection_per_tenant
  Why: Large tenants need index isolation. Collection per tenant.


---

## 8. Cost Optimization Tiers

Systematic approach to reducing RAG costs.

In [42]:
# Cost optimization priority order

COST_OPTIMIZATION_TIERS = {
    "tier_1_reduce_api_calls": {
        "priority": 1,
        "techniques": {
            "semantic_caching": "20-40% savings (60%+ for FAQ)",
            "prompt_caching": "50-90% on repeated prefixes",
            "response_caching": "100% on exact matches"
        },
        "implementation_effort": "Low",
        "demo_notebook": "litellm_demo.ipynb"
    },
    "tier_2_model_routing": {
        "priority": 2,
        "techniques": {
            "complexity_routing": "50-80% (simple → small model)",
            "local_embeddings": "98% savings vs API"
        },
        "implementation_effort": "Medium",
        "demo_notebook": "litellm_demo.ipynb"
    },
    "tier_3_token_reduction": {
        "priority": 3,
        "techniques": {
            "context_compression": "30-50% fewer tokens",
            "output_limits": "20-40% on output tokens",
            "chunking_optimization": "Variable"
        },
        "implementation_effort": "Medium",
        "demo_notebook": "chunking_strategies_demo.ipynb"
    },
    "tier_4_infrastructure": {
        "priority": 4,
        "techniques": {
            "batch_embeddings": "60% cost reduction",
            "quantization": "75% memory reduction"
        },
        "implementation_effort": "High",
        "demo_notebook": "embedding_models_demo.ipynb"
    }
}

print("Cost Optimization Priority")
print("=" * 60)
print("\nTypical RAG cost breakdown:")
print("  Embedding generation: 40-60%")
print("  Vector storage: 20-35%")
print("  LLM inference: 15-25%")
print("\n→ Optimize embedding costs first (highest impact)")
print("\nTiers (implement in order):")
for tier_name, tier_info in COST_OPTIMIZATION_TIERS.items():
    print(f"\n{tier_name.upper()}:")
    for technique, savings in tier_info["techniques"].items():
        print(f"  • {technique}: {savings}")
    print(f"  Effort: {tier_info['implementation_effort']}")

Cost Optimization Priority

Typical RAG cost breakdown:
  Embedding generation: 40-60%
  Vector storage: 20-35%
  LLM inference: 15-25%

→ Optimize embedding costs first (highest impact)

Tiers (implement in order):

TIER_1_REDUCE_API_CALLS:
  • semantic_caching: 20-40% savings (60%+ for FAQ)
  • prompt_caching: 50-90% on repeated prefixes
  • response_caching: 100% on exact matches
  Effort: Low

TIER_2_MODEL_ROUTING:
  • complexity_routing: 50-80% (simple → small model)
  • local_embeddings: 98% savings vs API
  Effort: Medium

TIER_3_TOKEN_REDUCTION:
  • context_compression: 30-50% fewer tokens
  • output_limits: 20-40% on output tokens
  • chunking_optimization: Variable
  Effort: Medium

TIER_4_INFRASTRUCTURE:
  • batch_embeddings: 60% cost reduction
  • quantization: 75% memory reduction
  Effort: High


---

## 9. Evaluation Metrics Reference

What to measure and target thresholds.

In [43]:
# RAG Evaluation Metrics

METRICS = {
    "retrieval": {
        "recall_at_k": {
            "definition": "Fraction of relevant docs in top K",
            "target": ">0.8",
            "measures": "Did we find the right documents?"
        },
        "precision_at_k": {
            "definition": "Fraction of top K that are relevant",
            "target": ">0.6",
            "measures": "Are retrieved docs actually useful?"
        },
        "mrr": {
            "definition": "Mean Reciprocal Rank of first relevant",
            "target": ">0.5",
            "measures": "How quickly do we find relevant docs?"
        }
    },
    "generation": {
        "faithfulness": {
            "definition": "Is answer supported by context?",
            "calculation": "(claims in context) / (total claims)",
            "target": ">0.8",
            "measures": "Hallucination prevention"
        },
        "answer_relevancy": {
            "definition": "Does answer address the question?",
            "target": ">0.7",
            "measures": "Response quality"
        },
        "context_precision": {
            "definition": "Are relevant chunks ranked higher?",
            "target": ">0.6",
            "measures": "Retrieval ranking quality"
        }
    }
}

print("RAG Evaluation Metrics")
print("=" * 60)
for category, metrics in METRICS.items():
    print(f"\n{category.upper()} METRICS:")
    for name, info in metrics.items():
        print(f"\n  {name}:")
        print(f"    Definition: {info['definition']}")
        print(f"    Target: {info['target']}")
        print(f"    Measures: {info['measures']}")

RAG Evaluation Metrics

RETRIEVAL METRICS:

  recall_at_k:
    Definition: Fraction of relevant docs in top K
    Target: >0.8
    Measures: Did we find the right documents?

  precision_at_k:
    Definition: Fraction of top K that are relevant
    Target: >0.6
    Measures: Are retrieved docs actually useful?

  mrr:
    Definition: Mean Reciprocal Rank of first relevant
    Target: >0.5
    Measures: How quickly do we find relevant docs?

GENERATION METRICS:

  faithfulness:
    Definition: Is answer supported by context?
    Target: >0.8
    Measures: Hallucination prevention

  answer_relevancy:
    Definition: Does answer address the question?
    Target: >0.7
    Measures: Response quality

  context_precision:
    Definition: Are relevant chunks ranked higher?
    Target: >0.6
    Measures: Retrieval ranking quality


---

## Summary

This notebook provides decision frameworks. For hands-on implementation:

| Topic | Notebook |
|-------|----------|
| Chunking strategies | `chunking_strategies_demo.ipynb` |
| Late chunking | `late_chunking_demo.ipynb` |
| Hybrid search + RRF | `hybrid_search_demo.ipynb` |
| Reranking | `reranking_demo.ipynb` |
| Embedding comparison | `embedding_models_demo.ipynb` |
| Qdrant patterns | `qdrant_demo.ipynb` |
| pgvector patterns | `pgvector_demo.ipynb` |
| Haystack RAG | `haystack_rag_demo.ipynb` |
| LangChain RAG | `langchain_rag_demo.ipynb` |
| LiteLLM routing | `litellm_demo.ipynb` |